# Data cleaning — 2025 HMDA Public LAR

Issue #1: prepare the notebook with the data cleaning script.

Source file: `2025_public_lar_csv.csv` (FFIEC HMDA public LAR, 2025), ~5.1 GB, 13,543,606 rows, 99 columns.
Not committed to git (see `.gitignore`) — place it at `data/2025_public_lar_csv.csv` in the repo root,
or point `DATA_PATH` below at wherever you keep it locally.

Missing values in the raw file are encoded as the literal string `"NA"`.

In [ ]:
import pandas as pd
from pathlib import Path

DATA_PATH = Path("../data/2025_public_lar_csv.csv")
RANDOM_STATE = 42

## 1. Load

Full file is 13.5M rows / 5 GB. Start from a sample while building the pipeline;
swap `nrows=SAMPLE_N` for a column subset (`usecols=`) or chunked read once the
cleaning steps below are finalized, to run on the full file.

In [ ]:
SAMPLE_N = 200_000  # set to None to load the full file

df = pd.read_csv(DATA_PATH, nrows=SAMPLE_N, na_values=["NA"], low_memory=False)
df.shape

## 2. Binary target

`action_taken` records the outcome of the application. Only codes `1` (loan
originated) and `3` (application denied) are clean accept/reject decisions;
the rest (withdrawn, incomplete, purchased loan, preapproval-only) don't
represent a credit decision and are dropped.

- `target = 1` → originated (`action_taken == 1`)
- `target = 0` → denied (`action_taken == 3`)

In [ ]:
df["action_taken"].value_counts()

In [ ]:
df = df[df["action_taken"].isin([1, 3])].copy()
df["target"] = (df["action_taken"] == 1).astype(int)

df["target"].value_counts(normalize=True)

## 3. Missingness

Check which columns are usable before deciding what to drop/impute.

In [ ]:
missing_pct = df.isna().mean().sort_values(ascending=False)
missing_pct[missing_pct > 0].to_frame("pct_missing")

*TODO: decide per-column handling here (drop columns above a missingness
threshold, impute numeric fields, encode categoricals) once we've agreed as a
team which predictors go into the model.*

## 4. Train / validation / test split

Single-year cross-sectional data (no temporal leakage risk), so a random
split stratified on `target` is sufficient — 70/15/15, fixed `random_state`
for reproducibility.

In [ ]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(
    df, test_size=0.30, stratify=df["target"], random_state=RANDOM_STATE
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, stratify=temp_df["target"], random_state=RANDOM_STATE
)

for name, split in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(f"{name}: {len(split):>8} rows, target=1 rate = {split['target'].mean():.3f}")